### Logic Gate Network

**Imports**

In [ ]:
import random
from scipy.special import softmax
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

**Logic operators**

In [2]:
def op_false(a, b): 
    return 0.0

def op_and(a, b): 
    return a * b

def op_not_imp_ab(a, b): 
    return a - a*b

def op_A(a, b): 
    return a

def op_not_imp_ba(a, b): 
    return b - a*b

def op_B(a, b): 
    return b

def op_xor(a, b): 
    return a + b - 2*a*b

def op_or(a, b): 
    return a + b - a*b

def op_not_or(a, b): 
    return 1 - (a + b - a*b)

def op_xnor(a, b): 
    return 1 - (a + b - 2*a*b)

def op_notB(a, b): 
    return 1 - b

def op_leqA(a, b): 
    return 1 - b + a*b

def op_notA(a, b): 
    return 1 - a

def op_implAB(a, b): 
    return 1 - a + a*b

def op_not_and(a, b): 
    return 1 - a*b

def op_true(a, b): 
    return 1.0

OPERATORS = [
    op_false, op_and, op_not_imp_ab, op_A,
    op_not_imp_ba, op_B, op_xor, op_or,
    op_not_or, op_xnor, op_notB, op_leqA,
    op_notA, op_implAB, op_not_and, op_true
]

**Logic Net**

In [3]:
class LogicNeuron:
    def __init__(self, in_features):
        self.i1 = random.randrange(in_features)
        self.i2 = random.randrange(in_features)

        self.logits = [random.uniform(-0.01, 0.01) for _ in range(16)]

    def forward(self, x):
        a = x[self.i1]
        b = x[self.i2]

        p = softmax(self.logits)

        return sum(p[k] * OPERATORS[k](a, b) for k in range(16))


class LogicLayer:
    def __init__(self, in_features, out_neurons):
        self.neurons = [LogicNeuron(in_features) for _ in range(out_neurons)]

    def forward(self, x):
        return [n.forward(x) for n in self.neurons]


class LogicNet:
    def __init__(self, input_bits, layers):
        self.layers = []
        in_f = input_bits
        for out_f in layers:
            self.layers.append(LogicLayer(in_f, out_f))
            in_f = out_f

    def forward(self, x):
        h = x
        for L in self.layers:
            h = L.forward(h)
        return h
    
    def predict(self, x):
        out = self.forward(x)
        return [1.0 if o >= 0.5 else 0.0 for o in out]
    
    def fit(self, X, Y, epochs=1000, lr=0.1):
        pass


**Test**

In [ ]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)

plt.figure(figsize=(6,6))
plt.scatter(X[:,0], X[:,1], c=y, cmap='bwr', edgecolor='k', s=40)
plt.title('make_moons dataset')
plt.xlabel('x1')
plt.ylabel('x2')
plt.gca().set_aspect('equal', 'box')
plt.show()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=25)

model = LogicNet(input_bits=2, layers=[4, 4, 1])
model.fit(X_train, y_train, epochs=1000, lr=0.1)
y_pred = model.predict(X_test)
accuracy = sum(1 for yt, yp in zip(y_test, y_pred) if yt == yp) / len(y_test)
print(f'Test accuracy: {accuracy * 100:.2f}%')
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)